In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.integrate

In [ ]:
flarndsim = np.load('/home/yousen/Public/ndlar_shared/data/larndsim_tred_comp20250606/larndsim_waveforms_unipolar_center.npz')

In [ ]:
flarndsim

In [ ]:
flarndsim.files

In [ ]:
print(flarndsim['segment_info'][0]['tran_diff'])
print(flarndsim['max_radius'])
print(flarndsim['max_pixels'])
flarndsim['max_neighboring_pixels']

In [ ]:
flarndsim['pixel_waveform'].shape

In [ ]:
np.argmax(flarndsim['pixel_waveform'].sum(axis=-1))

In [ ]:
ftred = np.load('/home/yousen/Public/ndlar_shared/data/larndsim_tred_comp20250606/single_segment_for_larndsim_unipolar_center.npz')
ftred.files

In [ ]:
ftred['current_tpc2_batch0_location']

In [ ]:
ttred = ftred['current_tpc2_batch0_location'][1,2]//2+np.arange(len(ftred['current_tpc2_batch0'][0, 0, 0, ::2]))
tlarndsim = np.arange(len(flarndsim['pixel_waveform'][0,4,::]))
tred_args = np.argsort(np.sum(np.squeeze(ftred['current_tpc2_batch0']), axis=-1))[::-1]
larndsim_args = np.argsort(np.sum(np.squeeze(flarndsim['pixel_waveform']), axis=-1))[::-1]
ctred = ftred['current_tpc2_batch0'][tred_args][:2].reshape(2, -1)
clarnd = np.squeeze(flarndsim['pixel_waveform'])[larndsim_args][:2].reshape(2, -1)
plt.plot(ttred, 1000*ctred[0].reshape(-1,2).sum(axis=-1), label='tred leading pixel')
plt.plot(ttred, 1000*ctred[1].reshape(-1,2).sum(axis=-1), label='tred subleading pixel')
plt.plot(tlarndsim, 0.1*clarnd[0], label='larnd-sim leading pixel')
plt.plot(tlarndsim, 0.1*clarnd[1], label='larnd-sim subleading pixel')
plt.legend()
plt.xlim(800, 1300)

In [ ]:
print(ftred['current_tpc2_batch0_location'][1,2])

In [ ]:
def id2pixel(pid):
    """
    Convert the unique pixel identifer to an x,y,plane tuple

    Args:
        pid (int): unique pixel identifier
    Returns:
        tuple: number of pixel pitches in x-dimension,
            number of pixel pitches in y-dimension,
            pixel plane number
    """
    npixels = (2*70, 4*70)
    return (pid % npixels[0], (pid // npixels[0]) % npixels[1],
            (pid // (npixels[0] * npixels[1])))

# at the time of saving, x, z were not swapped back
iystart = (flarndsim['segment_info'][0]['y_start'][0] - ftred['tpc_lower_left_tpc2'][0])/ftred['pixel_pitch_tpc2']
izstart = (flarndsim['segment_info'][0]['x_start'][0] - ftred['tpc_lower_left_tpc2'][1])/ftred['pixel_pitch_tpc2']
iyend = (flarndsim['segment_info'][0]['y_end'][0] - ftred['tpc_lower_left_tpc2'][0])/ftred['pixel_pitch_tpc2']
izend = (flarndsim['segment_info'][0]['x_end'][0] - ftred['tpc_lower_left_tpc2'][1])/ftred['pixel_pitch_tpc2']
print(iystart, iyend, izstart, izend)

pix_z, pix_y, pix_plane = id2pixel(flarndsim['unique_pixel_index'])

plt.plot(ftred['current_tpc2_batch0_location'][:,0], ftred['current_tpc2_batch0_location'][:,1], '*', label='induced pixels in tred')
plt.plot(pix_y[0]+0.1, pix_z[0]+0.1, 'o', label='induced pixels in larnd-sim')
plt.plot(np.unique(ftred['effq_tpc2_batch0_location'][:,:2], axis=0)[:,0]+0.2, np.unique(ftred['effq_tpc2_batch0_location'][:,:2], axis=0)[:,1]+0.2, '+', label='effq in tred')
plt.plot(np.unique(ftred['hits_tpc2_batch0_location'][:,:2], axis=0)[:,0]+0.3, np.unique(ftred['hits_tpc2_batch0_location'][:,:2], axis=0)[:,1]+0.3, 'x', label='hit in tred')
plt.plot([iystart, iyend], [izstart, izend], label='track segment')
plt.legend()

In [ ]:
print(np.max(ftred['current_tpc2_batch0'], axis=-1))

In [ ]:
print(np.min(ftred['current_tpc2_batch0'], axis=-1))

In [ ]:
print(ftred['current_tpc2_batch0_location'])

In [ ]:
print(np.sum(ftred['current_tpc2_batch0'], axis=-1))

In [ ]:
for i in range(len(np.squeeze(ftred['current_tpc2_batch0_location']))):
    x = np.squeeze(ftred['current_tpc2_batch0_location'])[i][2] + np.arange(len(np.squeeze(ftred['current_tpc2_batch0'])[i]))
    y = np.squeeze(ftred['current_tpc2_batch0'])[i]
    plt.plot(x, y, label=f"pixel, tick, at {np.squeeze(ftred['current_tpc2_batch0_location'])[i].tolist()}")
plt.xlim(1600, 2300)
plt.yscale('log')
plt.legend()

In [ ]:
np.sum(ftred['current_tpc2_batch0'])

In [ ]:
np.sum(flarndsim['pixel_waveform'], axis=-1)

In [ ]:
np.sum(flarndsim['pixel_waveform'])

In [ ]:
flarndsim['segment_info']['n_electrons']

In [ ]:
Hs = []
for i in range(5):  # or any subset of `i` values you'd like to compare
    xvals = (flarndsim['charge_samplings'][0, 0, 0, i, :, 0] - ftred['tpc_lower_left_tpc2'][1]) / 0.4434
    weights = flarndsim['charge_samplings'][0, 0, 0, i, :, 3]
    H, edges, _ = plt.hist(xvals, weights=weights, bins=100, alpha=0.6, label=f'Sample {i}', histtype='step', range=(100.2, 101.2))
    Hs.append(H)
print(np.max(np.abs(Hs[0]-Hs[1])))
print(np.max(np.abs(Hs[0]-Hs[2])))
print(np.max(np.abs(Hs[0]-Hs[3])))

plt.xlabel('Normalized X position')
plt.ylabel('Charge-weighted count')
plt.title('Charge distributions across different samples')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
Hs = []
for i in range(5):  # or any subset of `i` values you'd like to compare
    xvals = (flarndsim['charge_samplings'][0, 0, 0, i, :, 1] - ftred['tpc_lower_left_tpc2'][0]) / 0.4434
    weights = flarndsim['charge_samplings'][0, 0, 0, i, :, 3]
    H, edges, _ = plt.hist(xvals, weights=weights, bins=100, alpha=0.6, label=f'Sample {i}', histtype='step', range=(234.2, 235.7))
    Hs.append(H)
print(np.max(np.abs(Hs[0]-Hs[1])))
print(np.max(np.abs(Hs[0]-Hs[2])))
print(np.max(np.abs(Hs[0]-Hs[3])))

plt.xlabel('Normalized Y position')
plt.ylabel('Charge-weighted count')
plt.title('Charge distributions across different samples')
plt.legend()
plt.grid(True)
plt.tight_layout()

In [ ]:
Hs = []
xedges = []
yedges = []
for i in range(5):  # or any subset of `i` values you'd like to compare
    xvals = (flarndsim['charge_samplings'][0, 0, 0, i, :, 1] - ftred['tpc_lower_left_tpc2'][0]) / 0.4434
    yvals = (flarndsim['charge_samplings'][0, 0, 0, i, :, 0] - ftred['tpc_lower_left_tpc2'][1]) / 0.4434
    weights = flarndsim['charge_samplings'][0, 0, 0, i, :, 3]
    H, xes, yes = np.histogram2d(xvals, yvals, weights=weights, bins=(150, 100), range=((234.2, 235.7),(100.2, 101.2)))
    Hs.append(H)
    xedges.append(xes)
    yedges.append(yes)
print(np.max(np.abs(Hs[0]-Hs[1])))
print(np.max(np.abs(Hs[0]-Hs[2])))
print(np.max(np.abs(Hs[0]-Hs[3])))

In [ ]:
Hs = []
xedges = []
yedges = []
for i in range(12):  # or any subset of `i` values you'd like to compare
    xvals = (flarndsim['charge_samplings'][0, 0, i, 0, :, 1] - ftred['tpc_lower_left_tpc2'][0]) / 0.4434
    yvals = (flarndsim['charge_samplings'][0, 0, i, 0, :, 0] - ftred['tpc_lower_left_tpc2'][1]) / 0.4434
    weights = flarndsim['charge_samplings'][0, 0, i, 0, :, 3]
    H, xes, yes = np.histogram2d(xvals, yvals, weights=weights, bins=(150, 100), range=((234.2, 235.7),(100.2, 101.2)))
    Hs.append(H)
    xedges.append(xes)
    yedges.append(yes)
print(np.max(np.abs(Hs[0]-Hs[1])))
print(np.max(np.abs(Hs[0]-Hs[2])))
print(np.max(np.abs(Hs[0]-Hs[3])))

In [ ]:
for i in range(12):
    print('\nsum from sampling', np.sum(flarndsim['charge_samplings'][0, 0, i, 0, :, -1]), 'sum from histogram', np.sum(Hs[i]))
    qpxls = []
    for j in range(234,236):
        for k in range(100, 102):
            xarg_min = np.argmin(np.abs(xedges[0]-j))
            xarg_max = np.argmin(np.abs(xedges[0]-j-1))
            yarg_min = np.argmin(np.abs(yedges[0]-k))
            yarg_max = np.argmin(np.abs(yedges[0]-k-1))
            print(xarg_min, xarg_max, yarg_min, yarg_max)
            print('neighboring pixel', i, 'sum on induced pixel', j, k, np.sum(Hs[i][xarg_min:xarg_max, yarg_min:yarg_max]))
            qpxls.append(np.sum(Hs[i][xarg_min:xarg_max, yarg_min:yarg_max]))
    print('total from four pixels', np.sum(qpxls))

In [ ]:
plt.bar(xedges[0][:-1], np.sum(Hs[0], axis=-1), width=np.diff(xedges[0]), )


In [ ]:
plt.bar(xedges[0][:-1], np.sum(Hs[0], axis=-1), width=np.diff(xedges[0]), alpha=0.5)

uqlc = np.unique(ftred['effq_fine_grain_tpc2_batch0_location'][:,:1], axis=0)
for i in range(len(uqlc)):
    print(uqlc[i])
    lm = uqlc[i] == ftred['effq_fine_grain_tpc2_batch0_location'][:,:1]
    lm = lm.all(axis=1)
    print(ftred['effq_fine_grain_tpc2_batch0'][lm].shape)
    print(ftred['effq_fine_grain_tpc2_batch0'][lm].sum(axis=(0,2,3)))
    x = uqlc[i][0]+np.arange(0,1.0,0.1)
    y = ftred['effq_fine_grain_tpc2_batch0'][lm].sum(axis=(0,2,3))
    plt.plot(x, y*0.1)
    print(np.sum(y))
